In [7]:
import torch
import os
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import argparse
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)
from utils.f_utils import load_config, load_models, load_dataset
from models.medgemma_model import load_medgemma_model
from PIL import Image

# --------------
# Calcula Embeddings Para uma única imagem
# --------------
def generate_image_embedding(model, processor, image, device):

    with torch.no_grad():

        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        feats = model.get_image_features(pixel_values=inputs["pixel_values"])

        if isinstance(feats, dict):
            feats = feats.get("pooler_output", list(feats.values())[0])

        feats = feats / feats.norm(dim=-1, keepdim=True)

    return feats.squeeze(0).cpu()


class ImageClefDataset(Dataset):
    def __init__(self, split):
        """
        split: split do dataset (ex: ds["train"])
        """
        self.dataset = split

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        # Carrega a imagem apenas agora
        image = Image.open(sample["image_path"]).convert('RGB')

        data_dict = {
            "image": image,
            "caption": sample["caption"],
            "id": sample["image_id"]
        }

        return data_dict
    

In [11]:
config = load_config("../configs/rag_prompt_fine_tuned.yaml")
config

model, processor = load_medgemma_model(
            model_id=config["model"]["model_id"],
            use_quantization=config["model"]["use_quantization"],
            attn_implementation=config["model"]["attn_implementation"],
        )

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [15]:
config.get("MODEL_PATH", False)

'/home/ia368/projetos/imageclef2026-rag/artifacts/fine_tuning/medgemma-4b-it-sft-lora-5p-train-IC2026'

In [12]:
from peft import PeftModel

lora_path = config["MODEL_PATH"]  # your output_dir

model_lora = PeftModel.from_pretrained(model, lora_path)

/home/ia368/miniconda3/envs/rag_env/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [14]:
model_lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (embeddings): SiglipVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
              (position_embedding): Embedding(4096, 1152)
            )
            (encoder): SiglipEncoder(
              (layers): ModuleList(
                (0-26): 27 x SiglipEncoderLayer(
                  (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                  (self_attn): SiglipAttention(
                    (k_proj): lora.Linear4bit(
                      (base_layer): Linear4bit(in_features=1152, out_features=1152, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.05, inplace=False)
                      )
                  

In [13]:
model

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1152, out_features=16, bias=False)
                  )
   